# Grover's search on three quantum processors

| | |
|---|---|
| **Level** | Intermediate |
| **Time** | 60 to 90 minutes |
| **Prerequisites** | Grover's algorithm; what transpilation does |
| **Default devices** | Rigetti Cepheus and IQM Garnet (hardware); AQT IBEX Q1 (layout comparison only, no cost) |
| **Hardware jobs** | 4 per device |
| **Approximate cost** | about 210 credits on Garnet at 150 shots; Rigetti is billed by execution time, about 10 credits per job in our tests |

Shot counts for the hardware runs are set at the end of the **Setup** cell. The devices are set in the first hardware cell. Credits are charged only when a hardware cell runs.

*Part of the QUEST notebooks from qBraid: algorithms.*

On a simulator, Grover's algorithm returns the marked item with high probability. On real hardware the results are lower, and they differ between devices. This notebook runs one 4-qubit Grover circuit on real devices and looks at why the results differ.

We compare the measured success probabilities with the theoretical curve, and inspect what the compiler produced for each device. Transpiling is free, so the layout comparison also includes a trapped-ion device from AQT, which is not run on hardware by default.

**Learning objectives**

1. Build a standard 4-qubit Grover circuit.
2. Submit the same circuit to different devices through qBraid.
3. Compare measured success probabilities with the theoretical curve.
4. Read the transpiled circuit for each device, and see how the gate set and the qubit layout change it.
5. Relate hardware differences to algorithm design.

**Background needed:** Grover's algorithm, as covered in a first course. A short review follows.


## Grover's algorithm, the two-sentence version

Grover's algorithm finds a marked item in an unstructured database of $N$ items using approximately $\frac{\pi}{4}\sqrt{N}$ queries to an oracle, compared to $N/2$ on average classically. The trick is that we don't measure between queries. Instead we alternate an oracle (which flips the phase of the marked state) with a diffusion operator (which reflects amplitudes around their mean).

For $n$ qubits, $N = 2^n$. The success probability after $k$ Grover iterations is:

$$P_{\text{success}}(k) = \sin^2\left((2k + 1)\theta\right), \quad \text{where } \sin(\theta) = \frac{1}{\sqrt{N}}$$

For $n=4$ qubits ($N=16$), the optimal number of iterations is $\lfloor \frac{\pi}{4}\sqrt{16} \rfloor = 3$, at which point the success probability peaks near $96\%$ on an ideal quantum computer.

*The success probability is a smooth function of $k$. Run $k$ from 0 to 5 and you should see it rise, peak, and fall back down. That fall is the part worth watching for. It is the signature of Grover's algorithm working correctly.*


## Setup

In [ ]:
# Standard scientific Python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Qiskit for circuit construction and local simulation
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

# qBraid for unified device access
from qbraid.runtime import QbraidProvider

# Housekeeping
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 110
np.random.seed(42)  # reproducible randomization

print("Setup complete.")

# Shot counts for the hardware runs. More shots reduce statistical error but cost
# more on most devices; the README lists prices.
SHOTS = 150
QUEST_JOB_TAGS = {"quest": "algo-grover4"}   # labels this notebook's hardware jobs for QUEST usage statistics


## Building the circuit

We're going to mark the state $|1010\rangle$ (binary for the decimal number 10). The oracle will flip the phase of this state and leave all others untouched. The diffusion operator will then reflect amplitudes around the mean.

We build the oracle using a multi-controlled $Z$ gate sandwiched between $X$ gates on the qubits that should be $0$ in the target state. For $|1010\rangle$, qubits 0 and 2 should be $0$, so we apply $X$ gates to those before and after the MCZ.


In [ ]:
def grover_oracle_1010(qc):
    """Oracle marking |1010>. Qubit ordering: qc.qubits[0] is the rightmost bit."""
    # Apply X to qubits that should be 0 in |1010> (qubits 0 and 2)
    qc.x(0)
    qc.x(2)
    # Multi-controlled Z on 4 qubits
    qc.h(3)
    qc.mcx([0, 1, 2], 3)
    qc.h(3)
    # Undo the X gates
    qc.x(0)
    qc.x(2)
    return qc


def grover_diffusion(qc, n):
    """Standard diffusion operator (reflection around |+>^n)."""
    qc.h(range(n))
    qc.x(range(n))
    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)
    qc.h(n - 1)
    qc.x(range(n))
    qc.h(range(n))
    return qc


def build_grover_circuit(n=4, iterations=3):
    """Build a Grover circuit that searches for |1010> in n qubits."""
    qc = QuantumCircuit(n, n)
    # Uniform superposition
    qc.h(range(n))
    # Grover iterations
    for _ in range(iterations):
        grover_oracle_1010(qc)
        grover_diffusion(qc, n)
    # Measurement
    qc.measure(range(n), range(n))
    return qc


qc_test = build_grover_circuit(n=4, iterations=3)
qc_test.draw('mpl', fold=100)

Even at 4 qubits and 3 iterations the circuit is not small. Multi-controlled operations and Hadamard sandwiches accumulate quickly. This is why hardware matters here, since every extra gate is another chance for noise to eat your signal.

## Sanity check on the ideal simulator

Before we spend credits on real hardware, let's confirm the circuit does what we expect. We'll sweep the number of Grover iterations from 0 to 5 and plot the measured success probability. This should match the theoretical curve.


In [ ]:
def run_on_simulator(iterations, shots=4096):
    """Run Grover with `iterations` on the ideal simulator, return P(measure |1010>)."""
    qc = build_grover_circuit(n=4, iterations=iterations)
    sim = AerSimulator()
    result = sim.run(qc, shots=shots).result()
    counts = result.get_counts()
    return counts.get('1010', 0) / shots


iterations_range = range(0, 6)
sim_probs = [run_on_simulator(k) for k in iterations_range]

# Theoretical curve
theta = np.arcsin(1 / np.sqrt(16))
theory_probs = [np.sin((2 * k + 1) * theta) ** 2 for k in iterations_range]

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(iterations_range, theory_probs, 'k--', label='Theoretical', linewidth=2, alpha=0.6)
ax.plot(iterations_range, sim_probs, 'o-', color='#a02580', label='Aer simulator', markersize=10, linewidth=2)
ax.set_xlabel('Grover iterations')
ax.set_ylabel('P(measure |1010>)')
ax.set_title('Grover success probability on the ideal simulator')
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print(f"Peak success at k=3: {sim_probs[3]:.3f} (theory: {theory_probs[3]:.3f})")

The simulator matches theory almost exactly. Any tiny discrepancy is shot noise from a finite sample of 4,096 measurements per data point. This curve is the baseline. Anything we measure on real hardware that departs from it is a hardware effect, not an algorithm error.

## The devices

qBraid gives access to hardware from several vendors through one interface. This notebook runs on two superconducting devices and includes a trapped-ion device in the (free) layout comparison.

| Device | Vendor | Type | Qubits | Layout | Two-qubit gate |
|---|---|---|---|---|---|
| `rigetti:rigetti:qpu:cepheus-1-108q` | Rigetti | Superconducting | 107 | Square lattice | CZ |
| `aws:iqm:qpu:garnet` | IQM | Superconducting | 20 | Square lattice with clipped corners (30 connections) | CZ |
| `aws:aqt:qpu:ibex-q1` | AQT | Trapped ion | 12 | Every qubit connected to every other | not reported |

The main difference is the layout. In a trapped-ion device any qubit can interact with any other. In a superconducting device only neighbouring qubits can, so a two-qubit gate between distant qubits needs extra SWAP operations to bring them together. This routing often adds most of the extra circuit depth, and so most of the extra noise.

In [ ]:
# Connect to qBraid
provider = QbraidProvider()

# List available devices (for reference; comment this out once you know the IDs you want)
# devices = provider.get_devices()
# for d in devices:
#     print(f"{d.id}: status={d.status}, num_qubits={d.num_qubits}")

# For this notebook we target three specific devices.
# Devices are named by qBraid QRN. The README lists devices, prices and availability.
BACKENDS = {
    'Rigetti Cepheus': 'rigetti:rigetti:qpu:cepheus-1-108q',
    'IQM Garnet':      'aws:iqm:qpu:garnet',
    # 'AQT IBEX Q1':   'aws:aqt:qpu:ibex-q1',   # trapped ion; runs in scheduled windows; 2.35 credits per shot
}

# Compared in the transpilation table only. Transpiling is free; no job is submitted.
LAYOUT_ONLY = {'AQT IBEX Q1': 'aws:aqt:qpu:ibex-q1'}

# Colors for plotting
COLORS = {
    'Rigetti Cepheus': '#c63792',
    'IQM Garnet':      '#1a5285',
    'AQT IBEX Q1':     '#2d7a4f',
}

print("Configured backends:", list(BACKENDS.keys()))

## Transpilation: what runs on each device
Before submitting, let's look at what each vendor's compiler produces from the same logical circuit. To me this is the most instructive moment in the notebook. The same abstract algorithm becomes very different concrete gate sequences depending on the target hardware.


In [ ]:
# Device-specific transpilation.
#
# `device.profile.basis_gates` is None on every device except IonQ, so a plain
# `basis_gates=(... or None)` silently transpiles for no device at all and every backend
# below produces identical rows. On IonQ it raises instead, because its native names
# (gpi, gpi2, si, vi) are not Qiskit-standard. Every gate in HW_BASIS below IS in
# IonQ's accepted set, so this should work there, but it is not yet verified on IonQ.
#
# So: name the gate set explicitly, and take the qubit layout from `device.coupling_map` on
# newer qBraid SDKs, or from the device topology qBraid publishes on SDK 0.12. Trapped-ion machines report no coupling map because they are fully
# connected - no routing, no SWAPs - and that contrast is part of the lesson.
HW_BASIS = ['rz', 'rx', 'ry', 'cz', 'cx', 'h', 'measure']


def coupling_for(device):
    """Which qubits are connected, or None if every qubit connects to every other."""
    cmap = getattr(device, 'coupling_map', None)             # available on newer qBraid SDKs
    if cmap:
        return [list(edge) for edge in cmap]
    topology = provider.client.get_device(device.id).topology or {}   # qBraid SDK 0.12
    if 'fully' in topology.get('type', '') or not ('rows' in topology or 'rowSpans' in topology):
        return None
    spans = topology.get('rowSpans') or [[0, topology['cols'] - 1]] * topology['rows']
    index = {}
    for r, (c0, c1) in enumerate(spans):                      # number the grid sites row by row
        for c in range(c0, c1 + 1):
            index[(r, c)] = len(index)
    edges = []
    for (r, c), i in index.items():                           # connect each site to its right and lower neighbour
        for nb in ((r, c + 1), (r + 1, c)):
            if nb in index:
                edges += [[i, index[nb]], [index[nb], i]]
    return edges


def transpile_for(device, circuit, optimization_level=2):
    """Transpile for a specific device: its qubit layout and an explicit gate set."""
    return transpile(
        circuit,
        coupling_map=coupling_for(device),
        basis_gates=HW_BASIS,
        optimization_level=optimization_level,
    )


# Build the circuit we'll actually run
qc_grover = build_grover_circuit(n=4, iterations=3)

# Get device handles
devices = {name: provider.get_device(dev_id) for name, dev_id in BACKENDS.items()}

# Transpile for each backend and record stats
transpile_stats = []
layout_devices = {**devices, **{name: provider.get_device(q) for name, q in LAYOUT_ONLY.items()}}
for name, device in layout_devices.items():
    transpiled = transpile_for(device, qc_grover)
    stats = {
        'Backend': name,
        'Depth': transpiled.depth(),
        'Total gates': sum(transpiled.count_ops().values()),
        '2Q gates': sum(v for k, v in transpiled.count_ops().items() if k in ['cx', 'cz', 'iswap', 'ecr', 'rzx', 'xx']),
        'SWAPs inserted': transpiled.count_ops().get('swap', 0),
    }
    transpile_stats.append(stats)

# The comparison is the point of this notebook, so fail loudly if it stops being
# device-specific. This bug has appeared three times: first `qiskit_backend` (which was
# never a real field), then `basis_gates or None`, then `device.transform()` (a no-op).
if len(transpile_stats) > 1:
    _signatures = {(s['Depth'], s['Total gates'], s['2Q gates']) for s in transpile_stats}
    assert len(_signatures) > 1, (
        "every backend produced identical transpilation stats - the transpile is not "
        "device-specific. See FINDINGS.md 1c."
    )

# Pretty-print as a table
df = pd.DataFrame(transpile_stats).set_index('Backend')
df

Look at the differences carefully. The "2Q gates" column is usually the biggest driver of noise on today's hardware. Two-qubit gate fidelities are typically 99% at best, compared to 99.9% or better for single-qubit gates. If one backend needs twice as many two-qubit gates as another, you can predict which one will perform worse before you run it.

The "SWAPs inserted" column tells you how much the compiler had to work around limited connectivity. AQT (all-to-all) should show zero SWAPs. Rigetti and IQM will show some, and the exact number depends on which physical qubits the compiler chose to map your logical qubits onto.


## Submit to hardware

We'll run each backend at iteration counts $k \in \{1, 2, 3, 4\}$ so we can trace out the Grover curve on each device and compare it to the ideal curve. That's 4 circuits × 3 backends = 12 hardware jobs, each with 150 shots.

**Note on queue times.** Hardware jobs are queued, not instantaneous. Depending on device load this cell may take anywhere from a few minutes to a few hours to complete. The `job.wait_for_final_state()` calls block until each job finishes.


In [ ]:
ITERATIONS_TO_TEST = [1, 2, 3, 4]

hardware_results = {name: {} for name in BACKENDS}
jobs = {name: {} for name in BACKENDS}

# Submit all jobs
for name, device in devices.items():
    for k in ITERATIONS_TO_TEST:
        qc = build_grover_circuit(n=4, iterations=k)
        job = device.run(qc, shots=SHOTS, tags=QUEST_JOB_TAGS)
        jobs[name][k] = job
        print(f"Submitted: {name}, k={k}, job_id={job.id}")

# Wait for all jobs, collect results
for name in BACKENDS:
    for k in ITERATIONS_TO_TEST:
        job = jobs[name][k]
        result = job.result()
        counts = result.data.get_counts()
        # Normalize measurement bitstring format across vendors
        # (some return little-endian, some big-endian; qBraid provides a normalized interface)
        p_marked = counts.get('1010', 0) / SHOTS
        hardware_results[name][k] = p_marked
        print(f"{name}, k={k}: P(|1010>) = {p_marked:.3f}")

## Compare results

The devices are plotted against the ideal curve on the same axes.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Theoretical curve (fine grid)
k_fine = np.linspace(0, 5, 100)
theory_fine = np.sin((2 * k_fine + 1) * theta) ** 2
ax.plot(k_fine, theory_fine, 'k--', label='Ideal (theory)', linewidth=2, alpha=0.5)

# Simulator points (from earlier)
ax.plot(list(iterations_range), sim_probs, 'k.', markersize=8, alpha=0.6, label='Aer simulator')

# Hardware points
for name in BACKENDS:
    ks = list(hardware_results[name].keys())
    ps = [hardware_results[name][k] for k in ks]
    ax.plot(ks, ps, 'o-', color=COLORS[name], label=name,
            markersize=12, linewidth=2.5, markeredgecolor='white', markeredgewidth=1.5)

ax.set_xlabel('Grover iterations', fontsize=12)
ax.set_ylabel('P(measure |1010>)', fontsize=12)
ax.set_title('Grover on real hardware: three different QPUs vs ideal curve', fontsize=13, pad=15)
ax.set_ylim(0, 1.05)
ax.set_xticks(range(0, 6))
ax.grid(alpha=0.3)
ax.legend(loc='upper right', fontsize=11)
plt.tight_layout()
plt.show()

The simulator curve should track the theoretical prediction almost exactly. The hardware curves are all *below* the ideal, and they may not even peak at $k=3$. Noise may instead cause the success probability to saturate or even decrease as you add more iterations, because each additional iteration adds more noisy gates that degrade the state.

So on today's hardware, more Grover iterations does not mean better answers past a certain point. There is an optimal number of iterations *for your specific hardware*, and it may be less than the theoretical optimum. No ideal-simulator experiment can teach you that.

In [ ]:
# Detailed comparison table
comparison_data = []
for name in BACKENDS:
    row = {'Backend': name}
    for k in ITERATIONS_TO_TEST:
        row[f'k={k}'] = f"{hardware_results[name][k]:.3f}"
    row['Ideal (k=3)'] = f"{theory_probs[3]:.3f}"
    row['Fidelity ratio (k=3)'] = f"{hardware_results[name][3] / theory_probs[3]:.3f}"
    comparison_data.append(row)

pd.DataFrame(comparison_data).set_index('Backend')

## Discussion

The devices usually give noticeably different results. Two factors explain most of the difference:

1. **The number of two-qubit gates after transpilation.** More two-qubit gates means more accumulated error.
2. **The gate set and the qubit layout.** A device whose native gates fit the circuit poorly needs more decomposition, and a device with limited connections needs SWAPs to route qubits. Both add gates.

The ratio in the table above (measured probability at $k=3$ divided by the ideal probability) is a rough estimate of how much of the signal survives on each device. The ideal value is 0.96. A ratio near random guessing (0.0625 divided by 0.96) means the circuit was too long for the device as compiled; [the three-qubit Grover notebook](intro_02_grover_search_three_qubits.ipynb) runs a shorter, 3-qubit Grover circuit.

The ratio depends on the algorithm, the number of qubits and the gate pattern, so a different circuit could rank the devices differently.

## Design considerations for real hardware

Once you understand the differences, you can design better. A few practical patterns:

1. **Choose the modality that matches your algorithm structure.** Algorithms with high connectivity requirements (like some VQE ansatzes and QAOA on dense graphs) tend to run better on trapped-ion systems. Algorithms with local structure (like nearest-neighbor Ising simulation) can run efficiently on superconducting systems.

2. **Prefer shallower circuits when the mathematics allows.** For Grover, this means using fewer iterations than theory would suggest. For other algorithms, it means preferring approximate methods (e.g., low-depth variational circuits) over exact ones (e.g., deep QPE).

3. **Match the compiler's job.** Some compilers do heavy optimization; some do minimal. Passing `optimization_level=3` in Qiskit (or the equivalent in other frameworks) can cut gate counts, especially for high-connectivity circuits on limited-topology hardware.

4. **Benchmark before you commit.** Running a small version of your algorithm on all available hardware and looking at fidelity ratios is a cheap way to pick your target before scaling up. This notebook is essentially that benchmarking exercise for Grover.


## Going further

The same compare-across-devices workflow applies to any circuit.

- **Phase estimation on hardware.** The [phase estimation notebook](intermediate_02_phase_estimation_precision_vs_noise.ipynb) studies how precision and noise trade off, and adds zero-noise extrapolation.
- **Change the marked state.** Mark a different bitstring. Do the success rates change? Which bitstrings are hardest, and why?
- **Try mitigation.** Run Grover at $k=3$ with measurement error mitigation. How much of the ideal probability can you recover?
- **Run the layout comparison on hardware.** Uncomment AQT in `BACKENDS` when its scheduled window is open, and compare its result with its lower two-qubit gate count.